# 聚合 baseline comparison score shards

该 Notebook 只负责聚合某一个 baseline 的 shard score records, 不执行 baseline detector。

In [ ]:
# 可修改配置区
REPO_URL = "https://github.com/RICHAAARC/TSTW-VW.git"  # Colab 冷启动使用的仓库 clone URL
REPO_DIR = "/content/TSTW-VW"  # Colab 本地仓库根目录
GIT_REF = "explicit-sync"  # 使用的项目分支
DRIVE_MOUNT = "/content/drive"  # Google Drive 挂载点
DRIVE_RESULT_ROOT = "/content/drive/MyDrive/TSTW/results"  # Drive results 根目录
BASELINE_NAME = "external_hidden_framewise"  # 要聚合的 baseline 名称, 例如 external_videoseal/external_rivagan/external_hidden_framewise
BASELINE_SCORE_RECORD_PATHS = []  # 当前 baseline 所有 shard 的 baseline_formal_score_records.jsonl 路径
BASELINE_SCORE_AGGREGATION_RUN_ROOT = "/content/TSTW_runtime/runs/baseline_score_records_aggregation"  # 聚合本地 run_root
TARGET_FPR = 0.01  # 论文级 1% FPR 时使用 0.01
OVERWRITE_DRIVE_RESULT = True  # Drive 聚合目录已存在时是否覆盖


In [ ]:
from pathlib import Path
import json
import os
import re
import subprocess
import sys


def run_command(command, cwd=None):
    print("$", " ".join(map(str, command)), flush=True)
    completed = subprocess.run(command, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    if completed.stdout:
        print(completed.stdout, flush=True)
    if completed.stderr:
        print(completed.stderr, file=sys.stderr, flush=True)
    if completed.returncode != 0:
        raise RuntimeError("命令失败: " + " ".join(map(str, command)))
    return completed



def parse_json_process_stdout(stdout_text):
    """解析仓库脚本输出的 JSON。

    脚本通常使用 indent=2 打印多行 JSON, 因此不能只读取最后一行。
    该函数从完整 stdout 中定位第一个 JSON 对象并解析, 便于 Colab cell 稳定接收结果。
    """
    resolved_text = str(stdout_text).strip()
    if not resolved_text:
        raise ValueError("脚本 stdout 为空, 无法解析 JSON 结果。")
    json_start = resolved_text.find("{")
    if json_start < 0:
        raise ValueError("脚本 stdout 中未找到 JSON 对象。")
    return json.loads(resolved_text[json_start:])

def split_paths(raw_paths):
    if isinstance(raw_paths, (list, tuple)):
        return [str(item).strip() for item in raw_paths if str(item).strip()]
    return [item.strip() for item in re.split(r"[;,]", str(raw_paths)) if item.strip()]


## 1. 挂载 Drive 并获取仓库

In [ ]:
try:
    from google.colab import drive
    drive.mount(DRIVE_MOUNT)
except Exception as exc:
    print(f"非 Colab 环境或 Drive 挂载失败: {exc}")
repo_path = Path(REPO_DIR)
if repo_path.exists():
    run_command(["git", "fetch", "--depth", "1", "origin", GIT_REF], cwd=repo_path)
    run_command(["git", "checkout", GIT_REF], cwd=repo_path)
    run_command(["git", "pull", "--ff-only", "origin", GIT_REF], cwd=repo_path)
else:
    run_command(["git", "clone", "--depth", "1", "--branch", GIT_REF, REPO_URL, REPO_DIR])
short_commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR, text=True).strip()
print({"short_commit": short_commit})


## 2. 聚合 score records

In [ ]:
record_paths = split_paths(BASELINE_SCORE_RECORD_PATHS)
if not record_paths:
    raise ValueError("BASELINE_SCORE_RECORD_PATHS 不能为空。请填入该 baseline 所有 shard 的 score records 路径。")
aggregation_args = [
    sys.executable,
    "scripts/prepare_baselines/aggregate_baseline_score_records.py",
    "--run-root", BASELINE_SCORE_AGGREGATION_RUN_ROOT,
    "--result-root", DRIVE_RESULT_ROOT,
    "--baseline-name", BASELINE_NAME,
    "--target-fpr", str(TARGET_FPR),
    "--short-commit", short_commit,
]
if OVERWRITE_DRIVE_RESULT:
    aggregation_args.append("--overwrite")
for record_path in record_paths:
    aggregation_args.extend(["--record-path", record_path])
completed = run_command(aggregation_args, cwd=REPO_DIR)
aggregation_payload = parse_json_process_stdout(completed.stdout)
print(aggregation_payload)


## 3. 最终完成性判断

In [ ]:
materialized_path = Path(aggregation_payload["materialized_path"])
required_paths = {
    "aggregation_manifest": materialized_path / "artifacts" / "baseline_score_aggregation_manifest.json",
    "baseline_comparison_table": materialized_path / "tables" / "baseline_comparison_table.csv",
    "baseline_attack_breakdown": materialized_path / "tables" / "baseline_attack_breakdown.csv",
    "baseline_threshold_table": materialized_path / "tables" / "baseline_threshold_table.csv",
    "baseline_runtime_table": materialized_path / "tables" / "baseline_runtime_table.csv",
}
completion_summary = {
    "result_flow": "baseline_shard_aggregate",
    "baseline": BASELINE_NAME,
    "target_fpr": TARGET_FPR,
    "drive_aggregated_root": str(materialized_path),
    "input_record_paths": record_paths,
    "required_paths": {name: path.exists() for name, path in required_paths.items()},
    "ready_for_paper_artifact_gate": True,
}
completion_summary["status"] = all(completion_summary["required_paths"].values())
print(json.dumps(completion_summary, ensure_ascii=False, indent=2))
if not completion_summary["status"]:
    raise RuntimeError(completion_summary)
